# FinReason — SFT Training (Colab Pro A100)
Fine-tune Qwen2.5-7B on FinQA using QLoRA.

**Runtime:** Change to A100 GPU before running.
Runtime → Change runtime type → A100

In [ ]:
# Verify GPU
!nvidia-smi

In [ ]:
# Install dependencies
!pip install -q torch transformers datasets peft trl bitsandbytes accelerate wandb huggingface_hub PyYAML scipy

In [ ]:
# Mount Google Drive — checkpoints saved here
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/finreason/checkpoints/sft', exist_ok=True)
os.makedirs('/content/drive/MyDrive/finreason/data', exist_ok=True)

In [ ]:
# Clone repo
!git clone https://github.com/glenlouis8/finreason.git
%cd finreason

In [ ]:
# Login to HuggingFace using Colab secret
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get('HF_TOKEN'))

In [ ]:
# Login to W&B using Colab secret
import wandb
from google.colab import userdata
wandb.login(key=userdata.get('WANDB_API_KEY'))

In [ ]:
# Prepare SFT data
!python scripts/prepare_data.py --config configs/sft.yaml

# Copy data to Drive for reuse
!cp -r data/ /content/drive/MyDrive/finreason/data/

In [ ]:
# Override checkpoint output dir to save directly to Drive
import yaml

with open('configs/sft.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['training']['output_dir'] = '/content/drive/MyDrive/finreason/checkpoints/sft'

with open('configs/sft.yaml', 'w') as f:
    yaml.dump(cfg, f)

In [ ]:
# Run SFT training (~2-3 hours on A100)
!python scripts/train_sft.py --config configs/sft.yaml

In [ ]:
# Verify checkpoint saved
!ls /content/drive/MyDrive/finreason/checkpoints/sft/

## Done!
SFT checkpoint saved to Google Drive.

Next: open `colab_dpo.ipynb` to generate preference pairs and run DPO training.